# My Investigation into PageRank: What makes it tick?

I've always found PageRank fascinating because it turns a simple idea—"important pages are linked to by other important pages"—into a linear algebra problem. In this notebook, I'm going to explore how the **damping factor** $d$ actually behaves. I've heard it controls a trade-off between speed and accuracy, so I want to see that for myself.

## Setting the Scene

Mathematically, I'm looking for the stationary distribution of the Google Matrix $G$:
$$ G = d M + \frac{1-d}{n} J $$

I'm keeping the **ReCoDE Standard on Numerical Stability** in mind here. I learned that the convergence of power iteration is all about the **spectral gap**. If $d$ is the second-largest eigenvalue, then $1-d$ is the gap. If I push $d$ too close to $1$, that gap disappears. I want to see if I can actually measure the "computational penalty" of doing that.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

# Setting up my path so I can use my library code
sys.path.append(os.path.abspath(os.path.join('..')))

from src.linalg.pagerank import is_stochastic, pagerank

## 1. Testing the "Convergence Penalty"

I'll start by generating a random $100 \times 100$ matrix. I'll make sure it's stochastic first (using the validator I wrote), and then I'll loop through different values of $d$ to see how many iterations it takes to converge. I'm expecting the plots to get much longer as $d$ approaches 1.

In [ ]:
def generate_random_stochastic(n):
    M = np.random.rand(n, n)
    M /= M.sum(axis=0)
    return M

np.random.seed(42)
n_large = 100
M_large = generate_random_stochastic(n_large)

# Quick ReCoDE check: Did I build a valid matrix?
print(f"Is my test matrix valid/stochastic? {is_stochastic(M_large)}")

In [ ]:
damping_factors = [0.5, 0.7, 0.85, 0.9, 0.95, 0.99]
results = {}

plt.figure(figsize=(10, 6))

for d in damping_factors:
    # I'm tracking the 'history' to see how fast the error drops
    v, history = pagerank(M_large, d=d, tol=1e-10, return_history=True)
    plt.semilogy(history, label=f'd={d}')
    results[d] = len(history)

plt.xlabel('Iteration')
plt.ylabel('Residual (how much v is still changing)')
plt.title('My Experiment: Watching PageRank converge for different d')
plt.legend()
plt.grid(True, which="both", ls="-", alpha=0.5)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(list(results.keys()), list(results.values()), 'o-', color='darkred')
plt.xlabel('Damping Factor (d)')
plt.ylabel('Total Iterations')
plt.title('The "Penalty Plot": How much longer I have to wait as d increases')
plt.grid(True)
plt.show()

## 2. Double-Checking My Work: Is it actually stationary?

One thing I'm trying to be better at (the **Defensive Programming** standard) is not just assuming my code worked because it didn't crash. I want to take the vector $v$ I just found and check if multiplying it by $G$ actually gives me $v$ back.

In [ ]:
d_test = 0.85
v_res, _ = pagerank(M_large, d=d_test, tol=1e-12)

# I'll build G explicitly here just for this small-scale test
n = M_large.shape[0]
G_explicit = d_test * M_large + (1 - d_test) / n * np.ones((n, n))

# The stationary property check: Gv should be very close to v
diff = np.linalg.norm(G_explicit @ v_res - v_res)

print(f"Stationary Residual: {diff:.2e}")
print(f"Does it sum to 1? {np.sum(v_res):.4f}")

### A Note for My Future Self: Efficiency
I realized while writing this that building `G_explicit` (the $n \times n$ ones matrix) is pretty wasteful. If I were doing this for a massive dataset, I'd use the **vectorized trick** from the library: 
$$ Gv = d(Mv) + \frac{1-d}{n} \mathbf{1} $$
(since $\sum v_i = 1$). That's a much better way to follow the **Vectorization** standard.

## 3. Sanity Check: Does the ranking make sense?

To make sure I really understand what the ranks mean, I've constructed a tiny "toy web" with 4 nodes. 
- **C** is basically the "star" node because everyone links to it.
- **A** and **D** are more like starting points.
I want to see if the algorithm correctly identifies C as the most important.

In [ ]:
M_toy = np.array([
    [0,   0, 1, 0.25], # A (links from C, and some random jumps)
    [0.5, 0, 0, 0.25], # B (links from A)
    [0.5, 1, 0, 0.25], # C (links from A, B, and D - should be the winner!)
    [0,   0, 0, 0.25]  # D (mostly isolated)
])

nodes = ['A', 'B', 'C', 'D']
v_toy, _ = pagerank(M_toy, d=0.85)

plt.figure(figsize=(8, 5))
plt.bar(nodes, v_toy, color='skyblue', edgecolor='navy')
plt.ylabel('PageRank Score')
plt.title('My Toy Web Ranking')
plt.show()

for node, score in zip(nodes, v_toy):
    print(f"Node {node} Rank: {score:.4f}")

## My Reflections: The Damping Factor Trade-off

After playing around with this, the choice of $d=0.85$ makes a lot more sense to me now. 

When I set $d$ too low, the graph didn't seem to matter much—it was too much like a random walk. But when I pushed $d$ towards $0.99$, my "Penalty Plot" showed the iterations exploding. It's a classic case where **Numerical Stability** (the spectral gap) directly affects how much compute I have to use. 

By sticking with $0.85$, I'm getting a ranking that clearly favors the "hubs" (like Node C in my toy example) while keeping the math stable enough that it converges in a handful of steps. It's cool to see how a theoretical property like an eigenvalue gap turns into real-world waiting time for an algorithm to finish.